In [1]:
import json

In [2]:
from openpi.training.config import CalvinDataConfig
from openpi.policies.calvin_dataset import CalvinDataset

In [3]:
config = CalvinDataConfig(
    repo_path="../pace/openpi/data/calvin-task-ABC-D-lerobot"
)

dataset = CalvinDataset(config, 16)

Resolving data files:   0%|          | 0/18957 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/116 [00:00<?, ?it/s]

In [6]:
import json

import cv2
import einops
import mediapy

episode_info_fname = config.repo_path + "/meta/episodes.jsonl"
episode_info = {}
with open(episode_info_fname, 'r') as episode_file:
    for line in episode_file.readlines():
        data = json.loads(line)
        if len(data['tasks']) != 1:
            print(f"Episode {data['episode_idx']} has a weird number of tasks: {len(data['tasks'])} != 1")
        task_nl = data['tasks'][0]
        task_split = task_nl.split(':', 1)
        if len(task_split) != 2:
            print(f"Episode {data['episode_idx']} task split failed: `{task_nl}`")
        else:
            task_nl = task_split[1].strip()
        data['task_nl'] = task_nl
        episode_info[data['episode_index']] = data

def write_text_to_frame(frame, text):
    """Render text at the top-right of the agentview frame."""

    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = 0.4
    thickness = 1
    margin = 3
    (text_w, text_h), baseline = cv2.getTextSize(text, font, font_scale, thickness)

    x = max(margin, frame.shape[1] - text_w - margin)
    y = margin + text_h

    cv2.rectangle(
        frame,
        (max(0, x - 4), max(0, y - text_h - 4)),
        (min(frame.shape[1] - 1, x + text_w + 4), min(frame.shape[0] - 1, y + baseline + 4)),
        (0, 0, 0),
        -1,
    )
    cv2.putText(frame, text, (x, y), font, font_scale, (255, 255, 255), thickness, cv2.LINE_AA)

    return frame

def view_dataset(dataset, episode_idx, out="out"):
    start = dataset.episode_starts[episode_idx]
    end = dataset.episode_ends[episode_idx]
    print(f"Showing frames from {start} to {end}")
    images = []
    wrist_images = []
    task = episode_info[episode_idx]['task_nl']
    for i in range(start, end):
        data = dataset[i]
        image = einops.rearrange(data['observation/image'], 'c h w -> h w c').numpy().copy()
        image = cv2.resize(image, (400, 400), interpolation=cv2.INTER_NEAREST)
        write_text_to_frame(image, task)
        images.append(image)
        wrist_images.append(einops.rearrange(data['observation/wrist_image'], 'c h w -> h w c').numpy())
    mediapy.write_video(f"{out}.mp4", images, fps=10)
    mediapy.write_video(f"{out}_wrist.mp4", wrist_images, fps=10)

view_dataset(dataset, 0, "0")
view_dataset(dataset, 1, "1")

Showing frames from 0 to 43
Showing frames from 43 to 85


In [10]:
def gather_episodes(dataset, out="all", limit=None):
    images = []
    print(len(dataset.episode_starts), "episodes.")
    for episode_idx, start_idx in enumerate(dataset.episode_starts):
        if limit and episode_idx > limit:
            break
        task = episode_info[episode_idx]['task_nl']
        data = dataset[start_idx]
        image = einops.rearrange(data['observation/image'], 'c h w -> h w c').numpy().copy()
        image = cv2.resize(image, (400, 400), interpolation=cv2.INTER_NEAREST)
        write_text_to_frame(image, task)
        images.append(image)
    mediapy.write_video(f"{out}.mp4", images, fps=2)

gather_episodes(dataset, limit=100)

18957 episodes.
